# 02 Baseline 與 tag 過濾門檻

- **B0**：history 的 5 分比例（常數）。
- **B1**：`p_u = (Σ y_uj + β·p̄) / (|I_u| + β)`。
- **`max_df`**：拿掉出現在超過這個比例食譜中的 tags，用 J1（tags 的 Jaccard 加權平均）在小格點上選。

只用 **validation cold 列**選參數，結果存到 `results/params.json`（`BETA`、`MAX_DF`）。

In [1]:
%load_ext autoreload
%autoreload 2

import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import *
from features import (RecipeSets, build_features, group_columns, load_interactions, load_recipes,
                      make_snapshot, user_prior, weighted_average, year_snapshot)
from models import (VARIANTS, baseline_constant, fit_lr, lr_coefficients, predict, training_set,
                    tune_beta, tune_weighted_average)
from evaluation import auc, compare, evaluate, mean_ci, pretty
from ranking import ranking_metrics, sample_candidates

warnings.filterwarnings("ignore", category=FutureWarning)  # pandas copy-on-write notices
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
t0 = time.time()
train, val, test = load_interactions()
recipes = load_recipes()
print(f"loaded in {time.time() - t0:.0f}s: train {len(train):,}, validation {len(val):,}, test {len(test):,}, recipes {len(recipes):,}")

loaded in 8s: train 683,996, validation 22,933, test 22,224, recipes 178,265


In [3]:
MAX_DF = 0.5   # starting value; tuned below
rsets = RecipeSets(recipes)
snap_val = make_snapshot("validation", train, val, rsets, H=H, max_df=MAX_DF)
cold_val = snap_val.targets.cold.values == 1
y_val = snap_val.targets[f"y_{LABEL}"].values
u_val = snap_val.targets.u.values
print(f"validation snapshot: {snap_val.n:,} rows ({cold_val.sum():,} cold), {len(snap_val.pairs):,} pairs")

validation snapshot: 22,933 rows (6,410 cold), 1,749,741 pairs


In [4]:
beta_table = tune_beta(snap_val, LABEL, cold_val, betas=BETA_GRID)
BETA = float(beta_table.beta[0])
prior_val = user_prior(snap_val, LABEL, BETA)
print("BETA =", BETA)
beta_table

BETA = 1.0


,beta,AUC,LogLoss
0,1,0.7252,0.4722
1,2,0.7247,0.4718
2,5,0.7227,0.4724
3,10,0.7200,0.4735
4,20,0.7168,0.4751


In [5]:
rows = []
for md_ in MAX_DF_GRID:
    snap_val.compute_similarity(rsets, md_)
    best = tune_weighted_average(snap_val, "jac_tag", LABEL, prior_val, cold_val, ks=(10, 20, 50), alphas=(1, 2, 5)).iloc[0]
    rows.append({"max_df": md_, "k": best.k, "alpha": best.alpha, "AUC": best.AUC})
maxdf_table = pd.DataFrame(rows)
MAX_DF = float(maxdf_table.sort_values("AUC", ascending=False).max_df.iloc[0])
maxdf_table.to_csv(RESULTS / "tune_max_df.csv", index=False)
print("MAX_DF =", MAX_DF)
maxdf_table

MAX_DF = 0.35


,max_df,k,alpha,AUC
0,0.2000,50.0000,5.0000,0.7298
1,0.3500,50.0000,5.0000,0.7305
2,0.5000,50.0000,5.0000,0.7294
3,0.7500,50.0000,5.0000,0.7290
4,1.0000,50.0000,5.0000,0.7288


In [6]:
preds = {"B0": baseline_constant(snap_val, LABEL), "B1": prior_val}
res_base = evaluate({k: v[cold_val] for k, v in preds.items()}, y_val[cold_val], u_val[cold_val], n_boot=N_BOOT)
res_base.to_csv(RESULTS / "val_baselines.csv")
save_params(BETA=BETA, MAX_DF=MAX_DF)
pretty(res_base)

,AUC,LogLoss,Brier
model,,,
B0,"0.5000 [0.5000, 0.5000]","0.5331 [0.5046, 0.5637]","0.1743 [0.1613, 0.1881]"
B1,"0.7252 [0.6902, 0.7605]","0.4722 [0.4404, 0.5048]","0.1530 [0.1409, 0.1657]"
